# 00 Build the dataset

This notebook builds the dataset that the rest of the project uses. It only combines the sources. It does not clean or filter anything. All cleaning decisions are made in `01_preprocessing.ipynb`, after the data quality checks there.

**Why two sources are needed**

The project needs four kinds of data. Ratings for collaborative filtering, item text for content-based models, user attributes for demographic filtering and timestamps for context-aware models and time-based splits [3, slide 3]. No single MyAnimeList dataset has all four.

- The 2018 MyAnimeList Dataset [1] has ratings, user attributes (gender, location, birth date) and timestamps. It has no synopsis text.
- The 2020 Anime Recommendation Database [2] has a synopsis for each anime. It has no user attributes and no timestamps.

Both come from the same website and use the same anime ID. So the synopsis from the 2020 source can be joined to the 2018 data on that ID.

**Files used (in `data/raw`)**

| File | Source | Content |
|---|---|---|
| `users_cleaned.csv` | 2018 | one row per user |
| `animelists_cleaned.csv` | 2018 | one row per user and anime (score, status, dates) |
| `anime_cleaned.csv` | 2018 | one row per anime (metadata) |
| `anime_with_synopsis.csv` | 2020 | one row per anime (synopsis) |

**Output (in `data/merged`)**

- `users.parquet` with one row per user
- `anime.parquet` with one row per anime, metadata plus synopsis
- `interactions/` with one row per user and anime

In [1]:
import os
import pandas as pd

os.makedirs("../data/merged/interactions", exist_ok=True)

## 1. Users

Only the columns that describe the user are kept. The other columns in the file are totals calculated from the anime lists, which can be recalculated from the interactions if needed.

In [2]:
users = pd.read_csv(
    "../data/raw/users_cleaned.csv",
    usecols=["username", "user_id", "gender", "location", "birth_date", "join_date"],
)
users["birth_date"] = pd.to_datetime(users["birth_date"], errors="coerce")
users["join_date"] = pd.to_datetime(users["join_date"], errors="coerce")

print(users.shape)
users.drop(columns=["username"]).head()

(108711, 6)


,user_id,gender,location,birth_date,join_date
0,2255153,Female,"Chennai, India",1990-04-29,2013-03-03
1,37326,Male,"Detroit,Michigan",1991-08-01,2008-02-13
2,228342,Male,"Nayarit, Mexico",1990-12-14,2009-08-31
3,327311,Female,"Malaysia, Kuantan",1998-08-24,2010-05-10
4,5015094,Male,"Nijmegen, Nederland",1999-10-24,2015-11-26


## 2. Anime and synopsis

This is the join between the two sources. It is a left join on the anime ID, so every anime from the 2018 data is kept. An anime without a match in the 2020 file gets an empty synopsis.

In [3]:
anime = pd.read_csv(
    "../data/raw/anime_cleaned.csv",
    usecols=[
        "anime_id", "title", "title_english", "type", "source", "episodes",
        "rating", "score", "scored_by", "members", "popularity", "premiered",
        "studio", "genre", "duration_min", "aired_from_year",
    ],
)

synopsis = pd.read_csv(
    "../data/raw/anime_with_synopsis.csv",
    usecols=["MAL_ID", "sypnopsis"],
)
# The column name is misspelled in the original file
synopsis = synopsis.rename(columns={"MAL_ID": "anime_id", "sypnopsis": "synopsis"})

anime = anime.merge(synopsis, on="anime_id", how="left")

matched = anime["synopsis"].notna().sum()
print("Anime in the 2018 data:", len(anime))
print("Anime with a synopsis from the 2020 data:", matched)
print("Share matched:", round(matched / len(anime), 3))
anime.head(3)

Anime in the 2018 data: 6668
Anime with a synopsis from the 2020 data: 6323
Share matched: 0.948


,anime_id,title,title_english,type,source,episodes,rating,score,scored_by,popularity,members,premiered,studio,genre,duration_min,aired_from_year,synopsis
0,11013,Inu x Boku SS,Inu X Boku Secret Service,TV,Manga,12,PG-13 - Teens 13 or older,7.63,139250,231,283882,Winter 2012,David Production,"Comedy, Supernatural, Romance, Shounen",24.0,2012.0,chiyo Shirakiin is the sheltered daughter of a...
1,2104,Seto no Hanayome,My Bride is a Mermaid,TV,Manga,26,PG-13 - Teens 13 or older,7.89,91206,366,204003,Spring 2007,Gonzo,"Comedy, Parody, Romance, School, Shounen",24.0,2007.0,"During his summer vacation, middle school stud..."
2,5262,Shugo Chara!! Doki,Shugo Chara!! Doki,TV,Manga,51,PG - Children,7.55,37129,1173,70127,Fall 2008,Satelight,"Comedy, Magic, School, Shoujo",24.0,2008.0,Now Utau has left Easter and restarted her sin...


## 3. Interactions

The anime lists file is over 2 GB, so it is read in chunks of 2 million rows and each chunk is saved as its own parquet file.

Two things happen to each chunk.

- The date columns are converted to dates. Values that are not real dates, like `0000-00-00`, become empty.
- The username is replaced by the numeric `user_id` from the users table. This is an inner join, so a row is only kept if its user exists in the users table. Usernames are not saved.

User attributes and anime metadata are not copied into the interactions. They stay in their own tables and can be joined on `user_id` or `anime_id` when needed. This keeps the interactions small enough to load on a laptop.

In [4]:
reader = pd.read_csv(
    "../data/raw/animelists_cleaned.csv",
    usecols=[
        "username", "anime_id", "my_watched_episodes", "my_start_date",
        "my_finish_date", "my_score", "my_status", "my_last_updated",
    ],
    chunksize=2_000_000,
)

rows_read = 0
rows_saved = 0

for i, chunk in enumerate(reader):
    rows_read += len(chunk)

    chunk = chunk.rename(columns={
        "my_watched_episodes": "watched_episodes",
        "my_start_date": "start_date",
        "my_finish_date": "finish_date",
        "my_score": "score",
        "my_status": "status",
        "my_last_updated": "last_updated",
    })

    chunk["start_date"] = pd.to_datetime(chunk["start_date"], format="%Y-%m-%d", errors="coerce")
    chunk["finish_date"] = pd.to_datetime(chunk["finish_date"], format="%Y-%m-%d", errors="coerce")
    chunk["last_updated"] = pd.to_datetime(chunk["last_updated"], format="%Y-%m-%d %H:%M:%S", errors="coerce")

    chunk = chunk.merge(users[["username", "user_id"]], on="username", how="inner")
    chunk = chunk.drop(columns=["username"])
    chunk = chunk[[
        "user_id", "anime_id", "score", "status", "watched_episodes",
        "start_date", "finish_date", "last_updated",
    ]]

    chunk.to_parquet(f"../data/merged/interactions/part_{i:03d}.parquet", index=False)
    rows_saved += len(chunk)

print("Rows read:", rows_read)
print("Rows saved:", rows_saved)
print("Rows lost in the user join:", rows_read - rows_saved)

Rows read: 31284030
Rows saved: 31284030
Rows lost in the user join: 0


## 4. Save users and anime

In [5]:
users.drop(columns=["username"]).to_parquet("../data/merged/users.parquet", index=False)
anime.to_parquet("../data/merged/anime.parquet", index=False)

## 5. Check the result

The saved files are loaded again to confirm the sizes and to see how much of the data is covered by a synopsis.

In [6]:
interactions = pd.read_parquet("../data/merged/interactions", columns=["user_id", "anime_id"])

print("Interactions:", len(interactions))
print("Users:", interactions["user_id"].nunique())
print("Anime:", interactions["anime_id"].nunique())

in_anime_table = interactions["anime_id"].isin(anime["anime_id"])
has_synopsis = interactions["anime_id"].isin(anime.loc[anime["synopsis"].notna(), "anime_id"])

print("Share of interactions whose anime is in the anime table:", round(in_anime_table.mean(), 4))
print("Share of interactions whose anime has a synopsis:", round(has_synopsis.mean(), 4))

Interactions: 31284030
Users: 108710
Anime: 6668


Share of interactions whose anime is in the anime table: 1.0
Share of interactions whose anime has a synopsis: 0.9908


**Result of the merge**

- No interactions were lost in the user join. All 31,284,030 rows have a matching user.
- 6,323 of the 6,668 anime (94.8%) got a synopsis from the 2020 data. The anime without a match are mostly less popular ones, so they only cover 0.9% of the interactions. 99.1% of the interactions are on an anime with a synopsis.
- The users table has 108,711 users but only 108,710 appear in the interactions. One user has an empty anime list.

Two things to keep in mind for the rest of the project. The data ends in 2018, since the ratings come from the 2018 source. And `last_updated` is when the user last changed the entry, not when they watched or rated the anime.

In [7]:
pd.read_parquet("../data/merged/interactions/part_000.parquet").head()

,user_id,anime_id,score,status,watched_episodes,start_date,finish_date,last_updated
0,2255153,21,9,1,586,NaT,NaT,2013-03-03 10:52:53
1,2255153,59,7,2,26,NaT,NaT,2013-03-10 13:54:51
2,2255153,74,7,2,26,NaT,NaT,2013-04-27 16:43:35
3,2255153,120,7,2,26,NaT,NaT,2013-03-03 10:53:57
4,2255153,178,7,2,26,NaT,NaT,2013-03-27 15:59:13


## References

[1] Račinský, M. (2018). MyAnimeList Dataset. Kaggle. https://doi.org/10.34740/KAGGLE/DSV/45582

[2] Valdivieso, H. Anime Recommendation Database 2020. GitHub. https://github.com/Hernan4444/MyAnimeList-Database

[3] S18 slides. Theory of context-aware recommenders.